# Own Your Voice — start here

This ten-minute preflight checks Python, CUDA, Docker, and whether the Brev GPU supports the full three-lab path. Select the **Own Your Voice ASR Labs** kernel before continuing. Lab 3 records the separate **Own Your Voice Riva Client** kernel and should select it automatically when opened.

The notebooks use NVIDIA Parakeet CTC 0.6B. Lab 1 additionally requires a compute-capability 8.0+ GPU, NVIDIA AI Enterprise access, and a personal NGC API key. Lab 2 uses NVIDIA NeMo. Lab 3 requires NGC/Riva access and serves the selected `.nemo` model through Riva locally or on EKS. A T4 can run the reduced NeMo/Riva path, but not Speech NIM.

In [ ]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'voice_asr_lab').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'voice_asr_lab').exists():
    raise RuntimeError('Open this notebook from the workshop repository.')
sys.path.insert(0, str(ROOT / 'src'))
os.environ.setdefault('HF_HOME', str(ROOT / '.cache' / 'huggingface'))
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
print(f'Repository: {ROOT}')

In [ ]:
import json, shutil, subprocess
import torch
from voice_asr_lab.profiles import detect_profile, profile_table

assert torch.cuda.is_available(), 'CUDA is unavailable. Confirm the kernel and Brev GPU.'
props = torch.cuda.get_device_properties(0)
capability = torch.cuda.get_device_capability(0)
profile = detect_profile()
print(json.dumps({
    'gpu': props.name,
    'vram_gb': round(props.total_memory / 1024**3, 1),
    'cuda': torch.version.cuda,
    'compute_capability': f'{capability[0]}.{capability[1]}',
    'speech_nim_supported': capability[0] >= 8,
    'profile': profile.as_dict(),
    'docker': shutil.which('docker') is not None,
}, indent=2))

In [ ]:
import pandas as pd
pd.DataFrame(profile_table()).set_index('name')

In [ ]:
completed = subprocess.run(
    [sys.executable, str(ROOT / 'scripts' / 'preflight.py')],
    check=True,
    text=True,
    capture_output=True,
)
print(completed.stdout)

## Ready

If `speech_nim_supported` is `true`, continue to **Lab 1: NIM deployment** and have your NGC API key ready. If it is `false`, use a supported L4/A10/A100-class GPU for Lab 1 or skip to Lab 2. Lab 3 still requires Parakeet ASR NIM access.